In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from  xgboost import XGBRegressor


In [ ]:
#reading cleaned datasets ready for modeling
df = pd.read_csv('../data/processed/df.csv')
df_linear = pd.read_csv('../data/processed/df_linear.csv')
clv_df = pd.read_csv('../data/processed/clv_df.csv')

In [ ]:
df.info()
df.columns

**PART C**

**Baseline Model**

**model 1: Multilinear Regression**


In [ ]:

# -----------------------------
# 1. Select features
# -----------------------------
linear_features = [
    'age', 'gender', 'customer_segment', 'tenure_months',
    'avg_monthly_spend', 'purchase_frequency', 'num_products_owned',
    'engagement_score', 'num_support_calls', 'discount_usage_rate',
    'payment_method_missing', 'preferred_channel_missing',
    'discount_usage_rate_missing', 'recency_bucket'
]

target = 'customer_lifetime_value'

X = df_linear[linear_features]
y = df_linear[target]

# -----------------------------
# 2. Train-test split (FIRST)
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# -----------------------------
# 3. Define preprocessing
# -----------------------------
categorical_cols = ['gender', 'customer_segment', 'recency_bucket']
numeric_cols = [
    'age', 'tenure_months', 'avg_monthly_spend',
    'purchase_frequency', 'num_products_owned',
    'engagement_score', 'num_support_calls',
    'discount_usage_rate'
]

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first'), categorical_cols),
        ('num', StandardScaler(), numeric_cols)
    ],
    remainder='passthrough'  # keeps missingness flags
)

# -----------------------------
# 4. Build pipeline
# -----------------------------
model = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('regressor', LinearRegression())
])

# -----------------------------
# 5. Fit model
# -----------------------------
model.fit(X_train, y_train)

# -----------------------------
# 6. Evaluate
# -----------------------------
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_pred = model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R²:", r2_score(y_test, y_pred))


In [ ]:
import pandas as pd
import numpy as np

# 1. Access the trained regressor inside the pipeline
regressor = model.named_steps['regressor']

# 2. Access the preprocessor step to get the encoded feature names
preprocessor_step = model.named_steps['preprocess']
feature_names = preprocessor_step.get_feature_names_out()

# 3. Combine coefficients and names into a readable DataFrame
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': regressor.coef_,
    'Abs_Coefficient': np.abs(regressor.coef_)  # Useful for ranking impact
}).sort_values(by='Abs_Coefficient', ascending=False)

print(importance_df)


**Linear regression with log transformed target**

In [ ]:

# ----------------------------
# Log-transform target
# ----------------------------

y_train_log = np.log(y_train)
y_test_log = np.log(y_test)

# ---------------------------------------
# Fit model on log-transformed target
# ---------------------------------------

model.fit(X_train, y_train_log)

# ---------------------------------------
# Predict in log space
# ---------------------------------------

y_pred_log = model.predict(X_test)

# ---------------------------------------
# Convert back to original scale
# ---------------------------------------

y_pred = np.exp(y_pred_log) 


from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)


**Model 2: Decision Tree Regressor**

In [ ]:
df.info()

In [ ]:
tree_features = [
    'age', 'gender', 'customer_segment', 'tenure_months',
    'avg_monthly_spend', 'avg_transaction_value', 'purchase_frequency',
    'num_products_owned', 'engagement_score', 'churn_risk_score',
    'email_open_rate', 'num_support_calls', 
    'discount_usage_rate',

    # engineered numeric
    'spend_per_product', 'engagement_x_spend',

    # engineered categorical
    'recency_bucket',

    # missingness flags
    'discount_usage_rate_missing', 'days_since_last_purchase_missing',
    'satisfaction_score_missing', 'age_missing', 'churn_risk_missing',
    'avg_transaction_value_missing', 'num_support_calls_missing',
    'email_open_rate_missing', 'region_missing',
    'preferred_channel_missing', 'payment_method_missing'
]

target = 'customer_lifetime_value'

df_tree = df[tree_features + [target]].copy()


In [ ]:
#---------------------------------------------
# Label Encoding
#---------------------------------------------
from sklearn.preprocessing import LabelEncoder

cat_cols = ['gender', 'customer_segment', 'recency_bucket']

le = LabelEncoder()
for col in cat_cols:
    df_tree[col] = le.fit_transform(df_tree[col])


#----------------------------------------------
# Train-Test Split
#----------------------------------------------

X = df_tree.drop(columns=[target])
y = df_tree[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

#-----------------------------------------------
# Train Decision tree Rgressor
#-----------------------------------------------
from sklearn.tree import DecisionTreeRegressor

tree = DecisionTreeRegressor(
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42
)

#-----------------------------------------------
# Fit Decision Tree
#-----------------------------------------------
tree.fit(X_train, y_train)


#-----------------------------------------------
# Predict with Decision Tree
#-----------------------------------------------
y_pred = tree.predict(X_test)


#-----------------------------------------------
# Evaluate the model
#-----------------------------------------------

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R²:", r2_score(y_test, y_pred))

**Model 3: XGBoost**

In [ ]:
#---------------------------------------------------
# 1. Feature List (Tree-Friendly Dataset)
#---------------------------------------------------
tree_features = [
    'age', 'gender', 'customer_segment', 'tenure_months',
    'avg_monthly_spend', 'avg_transaction_value', 'purchase_frequency',
    'num_products_owned', 'engagement_score', 'churn_risk_score',
    'email_open_rate', 'num_support_calls', 
    'discount_usage_rate',

    # engineered numeric
    'spend_per_product', 'engagement_x_spend',

    # engineered categorical
    'recency_bucket',

    # missingness flags
    'discount_usage_rate_missing', 'days_since_last_purchase_missing',
    'satisfaction_score_missing', 'age_missing', 'churn_risk_missing',
    'avg_transaction_value_missing', 'num_support_calls_missing',
    'email_open_rate_missing', 'region_missing',
    'preferred_channel_missing', 'payment_method_missing'
]

target = 'customer_lifetime_value'

df_xgb = df[tree_features + [target]].copy()


#---------------------------------------------------
# 2. Label Encoding (XGBoost prefers this)
#---------------------------------------------------
from sklearn.preprocessing import LabelEncoder

cat_cols = ['gender', 'customer_segment', 'recency_bucket']

le = LabelEncoder()
for col in cat_cols:
    df_xgb[col] = le.fit_transform(df_xgb[col])


#---------------------------------------------------
# 3. Train-Test Split
#---------------------------------------------------
from sklearn.model_selection import train_test_split

X = df_xgb.drop(columns=[target])
y = df_xgb[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


#---------------------------------------------------
# 4. Train XGBoost Regressor
#---------------------------------------------------
from xgboost import XGBRegressor

xgb = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.01,
    max_depth=4,
    subsample=0.7,
    colsample_bytree=0.7,
    reg_lambda=1.0,
    reg_alpha=0.5,
    random_state=42,
    tree_method='hist'
    #early_stopping_rounds=50
)

xgb.fit(X_train, y_train)


#---------------------------------------------------
# 5. Predict
#---------------------------------------------------
y_pred = xgb.predict(X_test)


#---------------------------------------------------
# 6. Evaluate
#---------------------------------------------------
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R²:", r2_score(y_test, y_pred))


**Model 3B: XGBoost with gridsearch and log tranformed target**

In [ ]:
#---------------------------------------------------
# 1. Feature List (Tree-Friendly Dataset)
#---------------------------------------------------
tree_features = [
    
    'gender', 'customer_segment', 'tenure_months',
    'avg_monthly_spend', 'purchase_frequency',
    'num_products_owned', 'engagement_score', 'churn_risk_score',
    'email_open_rate', 
    #'num_support_calls', 
    #'discount_usage_rate',  #'age', 'avg_transaction_value'

    # engineered numeric
    'spend_per_product', 'engagement_x_spend',

    # engineered categorical
    'recency_bucket',

    # missingness flags
    'discount_usage_rate_missing', 'days_since_last_purchase_missing',
    'churn_risk_missing',
    'email_open_rate_missing'
      #'region_missing',
    #'preferred_channel_missing', 'payment_method_missing', 'age_missing', 
    #'satisfaction_score_missing', #'avg_transaction_value_missing', 'num_support_calls_missing',
]

target = 'customer_lifetime_value'



df_xgb_1 = df[tree_features + [target]].copy()

#---------------------------------------------------
# 3. Train-Test Split
#---------------------------------------------------
from sklearn.model_selection import train_test_split

X = df_xgb_1.drop(columns=[target])
y = df_xgb_1[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

y_train_log =np.log1p(y_train)
y_test_log = np.log1p(y_test)


In [ ]:
df

In [ ]:
#-----------------------------------------------------
#2. Define preprocessing 
#-----------------------------------------------------
from sklearn.model_selection import GridSearchCV, KFold

categorical_cols = ['gender', 'recency_bucket','customer_segment'] 
numerical_cols = [col for col in X.columns if col not in categorical_cols]

preprocessor = ColumnTransformer([
    ('num', 'passthrough', numerical_cols),
    ('cat', OneHotEncoder(handle_unknown = 'ignore', sparse_output = False), categorical_cols)
])

#------------------------------------------------------
#3. Build the pipeline
#------------------------------------------------------

pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('xgb', XGBRegressor(random_state = 42, tree_method = 'hist'))
])

#------------------------------------------------------
#4. small grid 
#------------------------------------------------------

param_grid = {
    'xgb__n_estimators' : [500],
    'xgb__max_depth' : [3, 4, 6],
    'xgb__learning_rate' : [0.01, 0.03, 0.1],
    'xgb__subsample' : [0.7, 0.8],
    'xgb__colsample_bytree' : [0.8]
}


#----------------------------------------------------
#5. grid search
#----------------------------------------------------

cv =KFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    estimator = pipe,
    param_grid = param_grid,
    cv = cv,
    scoring = 'neg_root_mean_squared_error',
    n_jobs = -1,
    verbose = 2
)

grid.fit(X_train, y_train_log)


#-------------------------------------------------
# 6. results
#-------------------------------------------------

print("Best parameters found: ", grid.best_params_)
print("Best cross-validation on log scale: ", -grid.best_score_)

#-------------------------------------------------
# 7. evaluate on test set
#-------------------------------------------------

best_model = grid.best_estimator_

y_pred_log = best_model.predict(X_test)
y_pred = np.expm1(y_pred_log)


print("Test R2: ", r2_score(y_test, y_pred))
print("Test RMSE: ", np.sqrt(mean_squared_error(y_test, y_pred)))
print("Test MAE: ", mean_absolute_error(y_test, y_pred))


#-------------------------------------------
#saving model for streamlit app
#-------------------------------------------
import joblib
joblib.dump(best_model, '../clv_model.pkl') #saving trained pipeline
print("Saved: clv_model.pkl")

#feature_names = best_model.named_steps['preprocessor'].get_feature_names_out()
#joblib.dump(feature_names, '../feature_names.pkl') #saving feature names

##print("Saved: clv_model.pkl + feature_names.pkl to repo root")
#print(f"Model expects {len(feature_names)} features")
#print("Feature names:", feature_names[:5], "...") #preview first 5 features



In [ ]:
# checking target distribution

fig, ax = plt.subplots(1,2, figsize=(12,5))
ax[0].hist(y, bins = 100); ax[0].set_title('Raw CLV)')
ax[1].hist(np.log1p(y), bins = 100); ax[1].set_title('Log CLV')
plt.show()

print('Zero CLV:', (y==0).mean())
print('CLV quantiles:', y.quantile([0.5, 0.9, 0.95, 0.99]))

**Point to Note**

After My consistent poor results esp the R2 of around 49%. I thought the problem could be my target variable. but the hist plot above tells a very different story. The RMSE too is not that bad. Thinking about it now an error of 4661 is about 16% if you ceck it against the median target value of around 29k. But the biggest challeng is R2. My model can only explain 49% of the data, that worse than guesss work.

- next step i will check feature importance to understand whats going on

In [ ]:
from xgboost import plot_importance

plot_importance(grid.best_estimator_.named_steps['xgb'], max_num_features = 15)
plt.show()

In [ ]:
#getting feature names after ColumnTransformer

feature_names = grid.best_estimator_.named_steps['preprocessor'].get_feature_names_out()

#get importances and sort
importances = grid.best_estimator_.named_steps['xgb'].feature_importances_
feat_imp = pd.Series(importances, index=feature_names).sort_values(ascending=False)

print(feat_imp.head(33))


**Additional Feature Engineering**

In [ ]:
df_uptd = df.copy()


df_uptd['monetary'] = df_uptd['avg_monthly_spend'] * df_uptd['tenure_months']
df_uptd['recency'] = clv_df['days_since_last_purchase'] #introduced days since last purchase back after dropping it


#-------------------------------------
#log tranform for skew
#-------------------------------------

df_uptd['log_recency'] = np.log(df_uptd['recency'] + 1)
df_uptd['log_monetary'] = np.log(df_uptd['monetary'] + 1)
df_uptd['log_purchase_frequency'] = np.log(df_uptd['purchase_frequency'] + 1)
df_uptd['log_avg_monthly_spend'] = np.log(df_uptd['avg_monthly_spend'] + 1)
df_uptd['log_tenure_months'] = np.log(df_uptd['tenure_months'] + 1)

#--------------------------------------
#RFM interactions
#--------------------------------------

df_uptd['R_x_F'] = df_uptd['recency'] * df_uptd['purchase_frequency']
df_uptd['R_x_M'] = df_uptd['recency'] * df_uptd['monetary']
df_uptd['F_x_M'] = df_uptd['purchase_frequency'] * df_uptd['monetary']

#--------------------------------------
#Log RFM interactions
#--------------------------------------

df_uptd['log_R_x_F'] = df_uptd['log_recency'] + df_uptd['log_purchase_frequency']
df_uptd['log_R_x_M'] = df_uptd['log_recency'] + df_uptd['log_monetary']
df_uptd['log_F_x_M'] = df_uptd['log_purchase_frequency'] + df_uptd['log_monetary']

#--------------------------------------
#value density features
#-------------------------------------

df_uptd['spend_per_product'] = df_uptd['avg_monthly_spend'] / df_uptd['num_products_owned'] + 1
df_uptd['purchases_per_tenure'] = df_uptd['purchase_frequency'] / df_uptd['tenure_months'] + 1

df_uptd = df_uptd.drop(columns = [
    'recency_bucket', 
    'gender', 
    'age', 
    'age_missing', 
    'region_missing',
    'preferred_channel_missing',
    'payment_method_missing',
    'avg_transaction_value'
    ])

**MoDel 3C XGBoost With New Engineered Features, RandomizedSearch**

In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

# -----------------------------
# 1. Features
# -----------------------------
tree2_features = [
    'tenure_months', 'purchase_frequency', 'avg_monthly_spend',
    'monetary', 'recency',
    'log_tenure_months', 'log_purchase_frequency', 'log_avg_monthly_spend',
    'log_monetary', 'log_recency',
    'R_x_F', 'R_x_M', 'F_x_M',
    'log_R_x_F', 'log_R_x_M', 'log_F_x_M',
    'engagement_x_spend', 'num_products_owned', 'spend_per_product', 'purchases_per_tenure',
    'customer_segment', 'churn_risk_score', 'engagement_score',
    'discount_usage_rate', 'email_open_rate', 'num_support_calls'
]

num_cols = [c for c in tree2_features if c != 'customer_segment']
cat_cols = ['customer_segment']

# -----------------------------
# 2. Preprocessor (NO SCALING)
# -----------------------------
preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(drop='first'), cat_cols)
], remainder='passthrough')

# -----------------------------
# 3. Pipeline
# -----------------------------
xgb_pipe = Pipeline([
    ('prep', preprocessor),
    ('xgb', XGBRegressor(
        objective='reg:squarederror',
        random_state=42,
        n_jobs=-1,
        eval_metric='rmse'
    ))
])

# -----------------------------
# 4. Param Grid
# -----------------------------

param_dist = {
    'xgb__n_estimators': [800, 1200, 1600, 2000],
    'xgb__learning_rate': [0.02, 0.03, 0.05, 0.1],
    'xgb__max_depth': [3, 4, 5, 6, 8],
    'xgb__min_child_weight': [1, 3, 5, 7],
    'xgb__gamma': [0, 0.1, 0.3, 0.5],
    'xgb__subsample': [0.7, 0.8, 0.9],
    'xgb__colsample_bytree': [0.7, 0.8, 0.9],
    'xgb__colsample_bylevel': [0.7, 0.9],
    'xgb__reg_alpha': [0, 0.1, 0.5],
    'xgb__reg_lambda': [1, 3, 5],
    'xgb__max_delta_step': [0, 1, 2]
}
                                                                # param_dist = {
                                                                #     'xgb__n_estimators': [1000],
                                                                #     'xgb__max_depth': [4, 5, 7],
                                                                #     'xgb__learning_rate': [0.03, 0.05, 0.1],
                                                                #     'xgb__subsample': [0.8],
                                                                #     'xgb__colsample_bytree': [0.8],
                                                                #     'xgb__reg_alpha': [0, 0.1],
                                                                #     'xgb__reg_lambda': [1, 5]
                                                                # }

# -----------------------------
# 5. Train/Test Split
# -----------------------------
X = df_uptd[tree2_features]
y = df_uptd['customer_lifetime_value']

y_log = np.log1p(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_log, test_size=0.2, random_state=42
)

# -----------------------------
# 6. Randomized Search
# -----------------------------

search = RandomizedSearchCV(
    xgb_pipe,
    param_dist,
    n_iter=25,          # <-- real search, not 3
    cv=3,
    scoring='r2',
    verbose=1,
    n_jobs=-1,
    random_state=42
)

                                                                           # search = RandomizedSearchCV(
                                                                            #     xgb_pipe,
                                                                            #     param_dist,
                                                                            #     n_iter=3,
                                                                            #     cv=3,
                                                                            #     scoring='r2',
                                                                            #     verbose=1,
                                                                            #     n_jobs=-1,
                                                                            #     random_state=42
                                                                            # )

search.fit(X_train, y_train)

# -----------------------------
# 7. Evaluate
# -----------------------------
y_pred_log = search.predict(X_test)
y_pred = np.expm1(y_pred_log)

r2 = r2_score(np.expm1(y_test), y_pred)
rmse = np.sqrt(mean_squared_error(np.expm1(y_test), y_pred))

print("Best params:", search.best_params_)
print("R²:", r2)
print("RMSE:", rmse)


#----------------------------------------
# feature importance
#----------------------------------------
best_pipe = search.best_estimator_

feature_names = best_pipe.named_steps['prep'].get_feature_names_out()
importance = best_pipe.named_steps['xgb'].feature_importances_

feat_imp = pd.Series(importance, index=feature_names).sort_values(ascending=False)

print("\nTop 10 features:")
print(feat_imp.head(10))





In [ ]:
from xgboost import plot_importance
import matplotlib.pyplot as plt

plot_importance(best_pipe.named_steps['xgb'], max_num_features=10)
plt.show()


In [ ]:
# tree2_features = [
#     #raw RFM (originals)
#     'tenure_months', 'purchase_frequency', 'avg_monthly_spend',

#     #engineered FRM core
#     'monetary', 'recency',

#     #log versions
#     'log_tenure_months', 'log_purchase_frequency', 'log_avg_monthly_spend',
#     'log_monetary', 'log_recency',

#     #interactions
#     'R_x_F', 'R_x_M', 'F_x_M',
#     'log_R_x_F', 'log_R_x_M', 'log_F_x_M',

#     #my existing winners in feature_importance
#     'engagement_x_spend', 'num_products_owned', 'spend_per_product', 'purchases_per_tenure',

#     #keep if not leakinng
#     'customer_segment', 'churn_risk_score', 'engagement_score',

#     # maybe keep
#     'discount_usage_rate', 'email_open_rate', 'num_support_calls'
# ]


In [ ]:
# numerical__cols = [
#     'tenure_months', 'purchase_frequency', 'avg_monthly_spend',
#     'monetary', 'recency',
#     'log_recency', 'log_purchase_frequency', 'log_monetary', 'log_tenure_months', 'log_avg_monthly_spend',
#     'R_x_F', 'R_x_M', 'F_x_M',
#     'log_R_x_F', 'log_R_x_M', 'log_F_x_M',
#     'engagement_x_spend', 'num_products_owned', 'spend_per_product', 'purchases_per_tenure',
#     'discount_usage_rate', 'email_open_rate', 'num_support_calls',
#     'engagement_score', 'churn_risk_score'
# ]

# category_cols = ['customer_segment']

# #---------------------------------------------
# # build preprocessor
# #---------------------------------------------

# preprocessor = ColumnTransformer([
#     ('num', StandardScaler(), numerical__cols),
#     ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), category_cols)
# ])

# #---------------------------------------------
# #3. full pipeline
# #---------------------------------------------

# xgb_pipe = pipeline([
#     ('preprocessor', preprocessor),
#     ('xgb', XGBRegressor(
#         objective = 'reg:squarederror',
#         random_state = 42,
#         n_jobs = -1,
#         early_stopping_rounds = 50,
#         eval_metric = 'rmse'
#     ))
# ])

# #--------------------------------------------
# #grid search
# #---------------------------------------------

# param_dist = {
#     'xgb__n_estimators' : [100],
#     'xgb__max_depth' : [4, 5, 7],
#     'xgb__learning_rate' : [0.03, 0.05, 0.1],
#     'xgb__subsample' : [0.8],
#     'xgb__colsample_bytree' : [0.8],
#     'xgb__reg_alpha' : [0, 0.1],
#     'xgb__reg_lambda' : [1, 5]
# }

# #----------------------------------------------
# #train/ test split
# #----------------------------------------------

# X= df_uptd[tree2_features]
# y = df_uptd['customer_lifetime_value']

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# #----------------------------------------------
# #log transform target variable
# #----------------------------------------------

# y_train_log = np.log1p(y_train)
# y_test_log = np.log1p(y_test)

# #---------------------------------------------
# #fit
# #---------------------------------------------

# search = RandomizedSearchCV(
#     xgb_pipe,
#     param_dist,
#     n_iter=3,
#     cv=3,
#     scoring='r2',
#     verbose=1,
#     n_jobs=-1,
#     random_state=42
# )

# search.fit(
#     X_train, y_train_log,
#     xgb__eval_set=[(search.estimator.named_steps['preprocessor'].fit_transform(X_test), y_test_log)],
#     xgb__verbose=False

# #-----------------------------------------
# #evaluate the model
# #-----------------------------------------

# y_pred_log = search.predict(X_test)
# y_pred = np.expm1(y_pred_log)

# r2 = r2_score(y_test, y_pred)
# rmse = np.sqrt(mean_squared_error(y_test, y_pred))

# print(f"Best params: {search.best_params_}")
# print(f"Test R2: {r2:.4f}")
# print(f"Test RMSE: {rmse:.4f}")


In [ ]:
# #--------------------------------------------
# #new feature importance
# #--------------------------------------------

# feature_names = search.best_estimator_.named_steps['preprocessor'].feature_names_out()
# importance = search.best_estimator_.named_steps['xgb'].feature_importances_
# feat_imp = pd.Series(importance, index=feature_names).sort_values(ascending=False)

# print("\n Top 20 features:")
# print(feat_imp.head(20))
